In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# ==========================================================
# Mask-Guided Variable Window Pooling (PyTorch version)
#   + expose last_hk and last_m for regularization
# ==========================================================

class MaskGuidedVariablePooling(nn.Module):
    def __init__(self, K, tau=1):
        super().__init__()
        self.K = K
        self.tau = tau

        self.h_mlp = nn.Sequential(
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

        # Exposed values for regularization
        self.last_hk = None   # [B]
        self.last_m  = None   # [B, W]

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        device = x.device

        # 1) Predict temporal horizon
        pooled = x.mean(dim=[2,3])          # [B, C]
        h_k = self.h_mlp(pooled).squeeze(1) # [B]

        # 2) Temporal positions
        t = torch.linspace(0.0, 1.0, W, device=device).unsqueeze(0)  # [1,W]

        # 3) Temporal mask
        m = torch.sigmoid(self.tau * (h_k.unsqueeze(1) - t))          # [B,W]

        # Save for regularization
        self.last_hk = h_k
        self.last_m  = m

        # 4) Slope
        s = torch.relu(m[:, :-1] - m[:, 1:])  # [B,W-1]

        eps = 1e-8
        p = (s + eps) / (s.sum(dim=1, keepdim=True) + eps)  # [B,W-1]
        cdf = torch.cumsum(p, dim=1)                         # [B,W-1]
        cdf[:, -1] = 1.0

        # 5) Quantile cuts
        quantiles = torch.linspace(0, 1, self.K+1, device=device)[1:-1]  # [K-1]

        cuts = []
        for q in quantiles:
            cut = torch.argmax((cdf >= q).int(), dim=1)  # [B]
            cuts.append(cut)

        if len(cuts) > 0:
            cuts = torch.stack(cuts, dim=1)              # [B,K-1]
        else:
            cuts = torch.empty(B,0,dtype=torch.long,device=device)

        # 6) Pooling
        outputs = []
        prev = torch.zeros(B, dtype=torch.long, device=device)
        idx = torch.arange(W, device=device).unsqueeze(0)  # [1,W]

        for r in range(self.K):
            if r < self.K-1:
                curr = cuts[:, r]
            else:
                curr = torch.full((B,), W-1, device=device, dtype=torch.long)

            seg_mask = (idx >= prev.unsqueeze(1)) & (idx <= curr.unsqueeze(1))
            seg_mask = seg_mask.float()

            weights = seg_mask * m
            weights = weights / (weights.sum(dim=1, keepdim=True) + eps)

            pooled_seg = (x * weights.unsqueeze(1).unsqueeze(2)).sum(dim=3)  # sum over W
            outputs.append(pooled_seg)

            prev = curr + 1

        output = torch.stack(outputs, dim=3)  # [B,C,H,K]
        return output


# ==========================================================
# MODEL
# ==========================================================

class Model(nn.Module):
    def __init__(self, n_classes=15):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, (1,3))
        self.pool1 = MaskGuidedVariablePooling(K=7)

        self.conv2 = nn.Conv2d(16, 16, (1,2))
        self.pool2 = MaskGuidedVariablePooling(K=5)

        self.flatten = nn.Flatten()

        dummy = torch.zeros(1,1,82,20)
        dummy = torch.relu(self.conv1(dummy))
        dummy = self.pool1(dummy)
        dummy = torch.relu(self.conv2(dummy))
        dummy = self.pool2(dummy)
        flatten_size = dummy.numel()

        self.fc = nn.Sequential(
            nn.Linear(flatten_size, 64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,n_classes)
        )

    def forward(self,x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x


# ==========================================================
# Regularization: class-aware hk separation (batch-level)
# ==========================================================

def hk_class_regularization(hk, y, eps=1e-8):
    """
    hk: [B] horizons
    y : [B] labels
    returns: (L_intra, L_inter)
      - L_intra: minimize within-class variance
      - L_inter: maximize between-class mean distances
    """
    unique = torch.unique(y)
    means = []
    intra = 0.0
    valid_classes = 0

    for c in unique:
        mask = (y == c)
        if mask.sum() >= 2:
            hk_c = hk[mask]
            mu = hk_c.mean()
            means.append(mu)
            intra = intra + hk_c.var(unbiased=False)
            valid_classes += 1
        elif mask.sum() == 1:
            # single sample class -> mean exists, variance 0
            means.append(hk[mask].mean())
            valid_classes += 1

    if valid_classes > 0:
        L_intra = intra / (valid_classes + eps)
    else:
        L_intra = hk.new_tensor(0.0)

    if len(means) >= 2:
        means = torch.stack(means)  # [Cbatch]
        # Pairwise squared distances
        diff = means.unsqueeze(0) - means.unsqueeze(1)
        dist2 = diff.pow(2)
        # remove diagonal
        L_inter = dist2.mean()
        # we want to MAXIMIZE inter-class distance -> so loss uses negative
    else:
        L_inter = hk.new_tensor(0.0)

    return L_intra, L_inter


# ==========================================================
# Entraînement (avec loss class-aware)
# ==========================================================

def train_model(X, Y, epochs=50, lambda_intra=0.10, lambda_inter=0.20):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device utilisé :", device)

    model = Model().to(device)
    
    # ----------------------------------------------------------
    # Nombre total de paramètres
    # ----------------------------------------------------------
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n==============================")
    print("Paramètres du modèle")
    print(f"Total paramètres      : {total_params:,}")
    print(f"Paramètres entraînables: {trainable_params:,}")
    print("==============================\n")

    X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    Y = torch.tensor(Y, dtype=torch.long)

    dataset = TensorDataset(X, Y)
    loader = DataLoader(dataset, batch_size=256, shuffle=True, pin_memory=True)

    optimizer = optim.Adam(model.parameters(), lr=5e-5)
    criterion = nn.CrossEntropyLoss()

    total_start = time.time()

    for epoch in range(epochs):
        epoch_start = time.time()
        model.train()

        running_loss = 0.0
        running_ce   = 0.0
        running_reg  = 0.0

        correct = 0
        total = 0

        progress_bar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for xb, yb in progress_bar:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(xb)

            # main CE loss
            ce = criterion(outputs, yb)

            # regularization on hk for pool1 & pool2
            hk1 = model.pool1.last_hk
            hk2 = model.pool2.last_hk

            L_intra1, L_inter1 = hk_class_regularization(hk1, yb)
            L_intra2, L_inter2 = hk_class_regularization(hk2, yb)

            # We want:
            # - minimize intra-class variance
            # - maximize inter-class distances (so subtract it)
            reg = (L_intra1 + L_intra2) * lambda_intra - (L_inter1 + L_inter2) * lambda_inter

            loss = ce + reg

            loss.backward()
            optimizer.step()

            bs = xb.size(0)
            running_loss += loss.item() * bs
            running_ce   += ce.item() * bs
            running_reg  += reg.item() * bs

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == yb).sum().item()
            total += bs

            progress_bar.set_postfix(loss=float(loss.item()),
                                    ce=float(ce.item()),
                                    reg=float(reg.item()))

        epoch_time = time.time() - epoch_start

        avg_loss = running_loss / total
        avg_ce   = running_ce / total
        avg_reg  = running_reg / total

        accuracy = 100 * correct / total
        speed = total / epoch_time

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"  Loss totale  : {avg_loss:.4f} (CE={avg_ce:.4f} + REG={avg_reg:.4f})")
        print(f"  Accuracy     : {accuracy:.2f}%")
        print(f"  Temps epoch  : {epoch_time:.2f}s")
        print(f"  Samples/sec  : {speed:.0f}")

        if torch.cuda.is_available():
            print(f"  GPU memory   : {torch.cuda.memory_allocated()/1024**2:.2f} MB")

    total_time = time.time() - total_start
    print(f"\nEntrainement terminé en {total_time/60:.2f} minutes")

    os.makedirs("./cicids_models", exist_ok=True)
    torch.save(model.state_dict(), "./cicids_models/model_dynamic_cicids.pth")

    return model

In [ ]:
# ==========================================================
# RUN (custom pooling)
# ==========================================================

print("--------------------Chargement données test--------------------")

X_input = np.load('./X_input_split_train_smote/X_input_0.npy')

for i in range(1, 20):
    X_input = np.concatenate((
        X_input,
        np.load(f'./X_input_split_train_smote/X_input_{i}.npy')
    ))

Y = np.load('./X_input_split_train_smote/Y.npy')

print("Shape X:", X_input.shape)
print("Shape Y:", Y.shape)

print("--------------------Entrainement CustomPooling streaming--------------------")
_ = train_model(
    X=X_input,
    Y=Y,
    lambda_intra = 0,
    lambda_inter=0
)